# 03 — Curate your own evaluation dataset

**Companion to Lesson 3.**

BEIR is a fixed benchmark — it tells you how a retrieval setup
does on *scientific claim verification* or *medical literature*
or whichever academic test set you picked. That's useful for
comparing approaches in the abstract. It is *not* useful for
answering the question your team actually has: "**does this
retrieve well on the kind of queries our users ask, against
the documents we actually have?**" For that you need a
domain-specific evaluation set built on your own corpus.

## What you'll build

A small **judgement list** (queries + relevance labels) over
the corpus you already ingested, using the standard
bootstrap pattern:

1. **Sample** documents from your corpus.
2. **Draft queries** with an LLM — one realistic query per
   doc, in the voice of an actual user.
3. **Pool candidates** — retrieve top-K for each draft query
   with whatever black box you want (here: hybrid α=0.8).
4. **Grade** the pooled `(query, doc)` pairs with an LLM
   judge on a 0–3 scale.
5. **Curate** — *you* spot-check and edit. The LLM is fast
   and consistent, but it doesn't know what *you* mean by
   relevant. The human-in-the-loop step is what makes the
   dataset trustworthy.
6. **Save & re-evaluate** — write the judgements to disk and
   rerun the metrics from notebook 01 against them.

> **Requires `OPENAI_API_KEY`** in `.env`. The full bootstrap
> on 20 docs ≈ 60 LLM calls — costs a few cents on
> `gpt-4o-mini`.

In [ ]:
import os, sys
# Notebooks live in agent-notebooks/. Add the repo root to the path so
# we can import the lab's library modules.
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

In [ ]:
DATASET   = "scifact"   # match what you ingested
N_DOCS    = 20          # sample size for the bootstrap
TOP_K     = 8           # candidates per query to grade
MODEL_LLM = "gpt-4o-mini"

import os
assert os.environ.get("OPENAI_API_KEY"), \
    "Set OPENAI_API_KEY in .env before running this notebook."

import pymongo
from lib import MONGODB_URI, DB_NAME, collection_name

client = pymongo.MongoClient(MONGODB_URI)
coll   = client[DB_NAME][collection_name(DATASET)]

## Step 1 — Sample documents from the corpus

We pull `N_DOCS` distinct parent documents from the ingested
chunks collection. Each will seed one (or more) draft
queries.

In [ ]:
import random
random.seed(42)

sample_docs = []
seen = set()
# Walk chunks; keep first chunk per parent doc until we have N_DOCS.
for row in coll.find({}, {"doc_id": 1, "title": 1, "text": 1}).limit(N_DOCS * 4):
    did = row["doc_id"]
    if did in seen:
        continue
    seen.add(did)
    sample_docs.append({
        "doc_id": did,
        "title" : row.get("title", ""),
        "text"  : row["text"][:1500],
    })
    if len(sample_docs) >= N_DOCS:
        break

print(f"Sampled {len(sample_docs)} documents.")
for d in sample_docs[:3]:
    print(f"  doc {d['doc_id']:<12}  {d['title'][:60]!r}")

## Step 2 — Draft queries with an LLM

We ask the LLM to write **one realistic query** that a
domain user might type to find each document. The prompt is
the lever here — phrase it for your real users. Want
keyword-style searches? Say so. Want long natural-language
questions? Say so.

In [ ]:
from openai import OpenAI
oai = OpenAI()

QUERY_PROMPT = (
    "You are helping build a retrieval evaluation set. Given a document, "
    "write ONE concise, realistic search query that a domain user might type "
    "to retrieve this exact document. The query should be 3-15 words and read "
    "naturally — not a paraphrase of the title. Do NOT include the document ID.\n\n"
    "Document title : {title}\n"
    "Document text  : {text}\n\n"
    "Output ONLY the query, no quotes or commentary."
)

draft_queries = []
for i, d in enumerate(sample_docs, 1):
    resp = oai.chat.completions.create(
        model=MODEL_LLM,
        messages=[{
            "role": "user",
            "content": QUERY_PROMPT.format(title=d["title"], text=d["text"]),
        }],
        temperature=0.7,
    )
    q = resp.choices[0].message.content.strip().strip('"').strip("'")
    draft_queries.append({"qid": f"my_q_{i:03d}", "query": q, "seed_doc_id": d["doc_id"]})
    print(f"  [{i:>2}/{len(sample_docs)}] {q}")

## Step 3 — Pool candidates for each draft query

For each draft query, retrieve the top-K with our hybrid
black box. The union of these results is the **pool** — the
set of `(query, doc)` pairs we'll judge. Pooling is the
standard IR evaluation pattern (TREC and BEIR both use it):
we don't grade every doc in the corpus, only the docs that
*some* retriever surfaced.

In [ ]:
import voyageai
from lib import MONGODB_BASE_URL, VOYAGE_API_KEY, embed_queries
from retrieve import hybrid

voyage = voyageai.Client(api_key=VOYAGE_API_KEY, base_url=MONGODB_BASE_URL)
q_texts = [q["query"] for q in draft_queries]
q_vecs  = embed_queries(voyage, q_texts)

pool: dict[str, list[dict]] = {}
for q, q_vec in zip(draft_queries, q_vecs):
    ranked = hybrid(coll, q_vec, q["query"], top_k=TOP_K, alpha=0.8)
    pool[q["qid"]] = ranked

total_pairs = sum(len(rows) for rows in pool.values())
print(f"Pooled {total_pairs} (query, candidate) pairs across {len(pool)} queries.")

## Step 4 — Grade with an LLM judge

For each `(query, candidate doc)` pair, ask the LLM to grade
relevance on the BEIR-standard 0–3 scale:

- **3** — highly relevant: directly answers the query
- **2** — relevant: clearly addresses the query topic
- **1** — marginally relevant: tangentially related
- **0** — not relevant

The LLM is consistent and cheap; the prompt below uses the
same wording the [TREC pooling tradition](https://trec.nist.gov/)
uses to keep grades comparable.

In [ ]:
import json
import re

JUDGE_PROMPT = (
    "You are an information retrieval relevance judge. Given a search "
    "query and a passage from a document, grade how relevant the passage "
    "is to the query on this scale:\n"
    "  3 - Highly relevant: directly answers the query.\n"
    "  2 - Relevant: clearly addresses the query topic.\n"
    "  1 - Marginally relevant: tangentially related.\n"
    "  0 - Not relevant.\n\n"
    "Query  : {query}\n"
    "Passage: {passage}\n\n"
    "Output ONLY a single JSON object: "
    '{{"grade": <0|1|2|3>, "reason": "<short>"}}'
)

def judge(query: str, passage: str) -> tuple[int, str]:
    resp = oai.chat.completions.create(
        model=MODEL_LLM,
        messages=[{"role": "user",
                   "content": JUDGE_PROMPT.format(query=query, passage=passage[:1500])}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data.get("grade", 0)), data.get("reason", "")

draft_qrels: dict[str, dict[str, int]] = {}
reasons:     dict[tuple[str, str], str] = {}
total = sum(len(v) for v in pool.values())
done = 0
for q in draft_queries:
    qid = q["qid"]
    draft_qrels[qid] = {}
    for row in pool[qid]:
        grade, reason = judge(q["query"], row["text"])
        draft_qrels[qid][row["doc_id"]] = grade
        reasons[(qid, row["doc_id"])] = reason
        done += 1
        if done % 10 == 0:
            print(f"  graded {done}/{total} pairs …")
print(f"  graded {done}/{total} pairs (done).")

## Step 5 — Curate (human in the loop)

This is the critical step. The LLM judge is a *draft*. Skim
the high-grade and low-grade ends — that's where mistakes
hide. Tweak any cell of `draft_qrels` directly. A small,
carefully reviewed set beats a large, blindly-trusted one.

In [ ]:
import pandas as pd

rows = []
for q in draft_queries:
    qid = q["qid"]
    for row in pool[qid]:
        did = row["doc_id"]
        rows.append({
            "qid"      : qid,
            "query"    : q["query"],
            "doc_id"   : did,
            "grade"    : draft_qrels[qid].get(did, 0),
            "reason"   : reasons.get((qid, did), ""),
            "title"    : row.get("title", "")[:60],
        })
review_df = pd.DataFrame(rows).sort_values(["qid", "grade"], ascending=[True, False])
review_df.head(20)

In [ ]:
# Want to override a judgement? Edit it directly. Example:
#   draft_qrels["my_q_001"]["1234"] = 2   # bump from 1 to 2
# Then re-run the cell above to see the updated table.

## Step 6 — Save and re-evaluate

Save the curated qrels alongside the draft queries, then
rerun the same evaluation we did in notebook 01 — this time
against *our* judgements.

In [ ]:
import pathlib, json

EVAL_DIR = pathlib.Path("../eval_sets")
EVAL_DIR.mkdir(exist_ok=True)
(EVAL_DIR / f"{DATASET}_custom_queries.json").write_text(
    json.dumps({q["qid"]: q["query"] for q in draft_queries}, indent=2),
    encoding="utf-8",
)
(EVAL_DIR / f"{DATASET}_custom_qrels.json").write_text(
    json.dumps(draft_qrels, indent=2),
    encoding="utf-8",
)
print(f"Wrote {EVAL_DIR.resolve()}/")

In [ ]:
from retrieve import text_only, vector_only, hybrid
from lib_metrics import compute_query_metrics, aggregate_metrics, format_summary

STRATEGIES = {
    "lexical (BM25)" : lambda q_vec, q_text: text_only(coll, q_text, top_k=TOP_K),
    "vector"         : lambda q_vec, q_text: vector_only(coll, q_vec, top_k=TOP_K),
    "hybrid α=0.8"   : lambda q_vec, q_text: hybrid(coll, q_vec, q_text, top_k=TOP_K, alpha=0.8),
}
for name, run in STRATEGIES.items():
    per_query = []
    for q, q_vec in zip(draft_queries, q_vecs):
        ranked = run(q_vec, q["query"])
        ranked_ids = [r["doc_id"] for r in ranked]
        per_query.append(compute_query_metrics(ranked_ids, draft_qrels[q["qid"]]))
    agg = aggregate_metrics(per_query)
    print(f"{name:<20}  {format_summary(agg)}")

## What's next

You've just built a tiny custom evaluation set. To make it
production-quality:

- **Scale up.** 20 queries is a smoke test. Aim for 100+
  before you trust the numbers to drive decisions.
- **Diversify queries.** Sample from real user logs if you
  have them. If not, prompt the LLM for multiple query
  *styles* per doc (keyword, natural-language, multi-hop).
- **Tighten the judging prompt.** Add a couple of in-prompt
  examples from your domain. Calibrate the LLM on a handful
  of human-graded examples before scaling.
- **Re-curate periodically.** Eval sets decay: corpora
  change, user needs change. Treat the eval set as code, not
  a finished artifact.

And once you have a trustworthy eval set, the **advanced**
lab in `phase4/` is where you'd go next — per-query alpha
routing, query rewriters (HyDE, decompose), cross-encoder
reranking. None of those are worth tuning without a solid
eval set to measure against. Which is exactly the point.